# Week 1 — OpenCV with CUDA on Google Colab (or your own NVIDIA GPU)

> *Google Colab üzerinde CUDA destekli OpenCV kurulumu (donanımınız destekliyorsa yerel ortamda VS Code ile de yapılabilir).*

**Why build from source?** The `opencv-python` / `opencv-contrib-python` wheels on PyPI (and the ones Colab pre-installs) are compiled **without CUDA**, so `cv2.cuda.getCudaEnabledDeviceCount()` returns `0` even on a GPU runtime. The GPU modules (`cudaarithm`, `cudafilters`, `cudaimgproc`, …) live in the separate **opencv_contrib** repository and must be compiled with NVIDIA's `nvcc`.

**Before you run:** on Colab, `Runtime → Change runtime type → T4 GPU`. Locally (Linux or WSL2 with an NVIDIA driver and the CUDA Toolkit), select your Python environment as the kernel; nothing needs root.

What this notebook does:

1. Checks the GPU and its *compute capability* (T4 = 7.5, RTX 5070 Ti = 12.0).
2. On Colab, optionally mounts Google Drive — if a previous build is cached there, it is restored in seconds.
3. Otherwise clones OpenCV + opencv_contrib and builds **only the modules we need**, for **only this GPU's architecture** (this is what keeps the build to minutes instead of hours).
4. Installs the result (Colab: `/opt/opencv-cuda`, locally: `~/opencv-cuda`), hides CPU-only wheels, and verifies `cv2.cuda` works.

The saved outputs below come from a **local run** (RTX 5070 Ti, WSL2 Ubuntu 24.04, CUDA 13.0). The Colab T4 run is documented in the README.

## 1. Check the GPU

In [1]:
import os, shutil

# Local CUDA installs often leave nvcc off the PATH that Jupyter sees; Colab already has it.
cuda_home = os.environ.get("CUDA_HOME") or ("/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else None)
if shutil.which("nvcc") is None and cuda_home:
    os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]

!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
!nvcc --version | tail -n 2

name, compute_cap, driver_version, memory.total [MiB]
NVIDIA GeForce RTX 5070 Ti, 12.0, 616.92, 16303 MiB


Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [2]:
import subprocess, sys

OPENCV_VERSION = "5.0.0"   # same tag for opencv and opencv_contrib

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab: sources in /content, install in /opt (we are root there).
# Locally: everything under the home folder, so no sudo is needed.
WORK = "/content" if IN_COLAB else os.path.expanduser("~/opencv-cuda-src")
PREFIX = "/opt/opencv-cuda" if IN_COLAB else os.path.expanduser("~/opencv-cuda")
os.makedirs(WORK, exist_ok=True)

# Tools installed with pip (cmake, ninja) land next to this Python; make sure the shell finds them.
os.environ["PATH"] = os.path.dirname(sys.executable) + ":" + os.environ["PATH"]

try:
    cc = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).split()[0]
except Exception as e:
    raise SystemExit("No NVIDIA GPU found. Colab: Runtime → Change runtime type → T4 GPU, then run again.") from e

PYVER = f"{sys.version_info.major}.{sys.version_info.minor}"
print("Compute capability:", cc, "| Python", PYVER, "| OpenCV tag", OPENCV_VERSION)
print("Environment:", "Google Colab" if IN_COLAB else "local machine", "| install prefix:", PREFIX)

Compute capability: 12.0 | Python 3.12 | OpenCV tag 5.0.0
Environment: local machine | install prefix: /home/iliya/opencv-cuda


## 2. (Optional, Colab) Cache on Google Drive

The build is slow; the result is not. On Colab, set `USE_DRIVE = True` to save the finished build as a `.tar.gz` in your Drive. Next time (new runtime, same GPU type, same Python version) this notebook restores it instead of compiling. The cache name includes the OpenCV version, GPU architecture and Python version, so a mismatch simply triggers a fresh build. Locally there is nothing to cache: the build stays in `~/opencv-cuda`, and the next cell skips the build when it is already there.

In [3]:
USE_DRIVE = False

CACHE = None
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/opencv_cuda_cache", exist_ok=True)
    CACHE = f"/content/drive/MyDrive/opencv_cuda_cache/opencv-{OPENCV_VERSION}-sm{cc.replace('.', '')}-py{PYVER}.tar.gz"
    print("Cache file:", CACHE, "(exists)" if os.path.exists(CACHE) else "(not built yet)")

## 3. Build (or restore)

Key CMake options:

| Option | Why |
| --- | --- |
| `WITH_CUDA=ON` | compile the CUDA code paths |
| `OPENCV_EXTRA_MODULES_PATH=…/opencv_contrib/modules` | the `cuda*` modules live in contrib |
| `BUILD_LIST=core,imgproc,imgcodecs,python3,cudev,cudaarithm,cudafilters,cudaimgproc` | build only what we use → much faster |
| `CUDA_ARCH_BIN=<cc>` | generate GPU code only for this card (T4 → `7.5`, RTX 5070 Ti → `12.0`) |
| `WITH_CUDNN=OFF`, `OPENCV_DNN_CUDA=OFF` | the DNN module is not needed this week |
| `OPENCV_PYTHON3_INSTALL_PATH` | put the `cv2` package inside our own prefix so it can be cached |

The build log is written to `opencv_build.log` in the work folder; only the tail is printed. Each `nvcc` job can use 1–2 GB of RAM, so the number of parallel jobs is limited by memory as well as by CPU cores. Measured on a free Colab T4 runtime (2 CPU cores, CUDA 13.0, Python 3.13): **43 minutes**; the local build time is printed below. The CUDA filter modules take most of the time.

In [4]:
import time, multiprocessing

def sh(cmd, log=None):
    print("$", cmd if len(cmd) < 160 else cmd[:157] + "...")
    if log:
        cmd = f"({cmd}) >> {log} 2>&1"
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode != 0:
        if log:
            subprocess.run(f"tail -n 60 {log}", shell=True)
        raise RuntimeError(f"command failed ({r.returncode})")

def parallel_jobs():
    with open("/proc/meminfo") as f:
        mem_gb = int(f.readline().split()[1]) / 2**20          # MemTotal, kB → GB
    return max(1, min(multiprocessing.cpu_count(), int(mem_gb / 1.5)))

if os.path.isdir(f"{PREFIX}/python/cv2"):
    print("Already built and installed in", PREFIX, "→ skipping the build.")
elif CACHE and os.path.exists(CACHE):
    sh(f"mkdir -p {PREFIX} && tar -xzf '{CACHE}' -C {PREFIX}")
    print("Restored from Drive cache.")
else:
    t0 = time.time()
    LOG = f"{WORK}/opencv_build.log"
    open(LOG, "w").close()
    if IN_COLAB:
        sh("apt-get -qq update && apt-get -qq install -y ninja-build", LOG)
    sh(f"{sys.executable} -m pip -q install --upgrade cmake ninja", LOG)
    os.chdir(WORK)
    for repo in ("opencv", "opencv_contrib"):
        if not os.path.isdir(repo):
            sh(f"git clone -q --depth 1 --branch {OPENCV_VERSION} https://github.com/opencv/{repo}.git", LOG)

    import numpy
    cmake = f"""cmake -S {WORK}/opencv -B {WORK}/opencv/build -G Ninja \
      -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX={PREFIX} \
      -DOPENCV_EXTRA_MODULES_PATH={WORK}/opencv_contrib/modules \
      -DBUILD_LIST=core,imgproc,imgcodecs,python3,cudev,cudaarithm,cudafilters,cudaimgproc \
      -DWITH_CUDA=ON -DCUDA_ARCH_BIN={cc} -DCUDA_ARCH_PTX= \
      -DWITH_CUDNN=OFF -DOPENCV_DNN_CUDA=OFF -DCUDA_FAST_MATH=ON -DWITH_CUBLAS=ON \
      -DBUILD_TESTS=OFF -DBUILD_PERF_TESTS=OFF -DBUILD_EXAMPLES=OFF -DBUILD_DOCS=OFF \
      -DBUILD_opencv_apps=OFF -DBUILD_JAVA=OFF -DWITH_QT=OFF -DWITH_GTK=OFF \
      -DPYTHON3_EXECUTABLE={sys.executable} \
      -DPYTHON3_NUMPY_INCLUDE_DIRS={numpy.get_include()} \
      -DOPENCV_PYTHON3_INSTALL_PATH={PREFIX}/python"""
    sh(cmake, LOG)
    jobs = parallel_jobs()
    print(f"Building with {jobs} parallel jobs ({multiprocessing.cpu_count()} CPU threads)")
    sh(f"cmake --build {WORK}/opencv/build -j{jobs}", LOG)
    sh(f"cmake --install {WORK}/opencv/build", LOG)
    print(f"Built in {(time.time() - t0) / 60:.1f} min")
    if CACHE:
        sh(f"tar -czf '{CACHE}' -C {PREFIX} .")
        print("Saved cache to Drive:", CACHE)

$ /home/iliya/deep_venv/bin/python -m pip -q install --upgrade cmake ninja


$ git clone -q --depth 1 --branch 5.0.0 https://github.com/opencv/opencv.git


$ git clone -q --depth 1 --branch 5.0.0 https://github.com/opencv/opencv_contrib.git


$ cmake -S /home/iliya/opencv-cuda-src/opencv -B /home/iliya/opencv-cuda-src/opencv/build -G Ninja       -DCMAKE_BUILD_TYPE=Release       -DCMAKE_INSTALL_PREFI...


Building with 10 parallel jobs (20 CPU threads)
$ cmake --build /home/iliya/opencv-cuda-src/opencv/build -j10


$ cmake --install /home/iliya/opencv-cuda-src/opencv/build


Built in 10.2 min


## 4. Make Python use the CUDA build

Colab ships several CPU-only OpenCV wheels. If they stay installed, `import cv2` may load one of them instead of ours, so we uninstall them and register our build with a `.pth` file (a one-line text file that adds a folder to Python's import path). Locally the same happens inside the environment that runs this notebook (a virtual environment is recommended).

On Colab the `.pth` file is lost when the runtime is recycled, so **run this cell (and the cache restore above) once per new runtime**. After this cell, restart the session only if `cv2` was already imported earlier in it.

In [5]:
sh(f"{sys.executable} -m pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless || true")
import site
pth = os.path.join(site.getsitepackages()[0], "opencv_cuda.pth")
with open(pth, "w") as f:
    f.write(f"{PREFIX}/python\n")
print("wrote", pth)

$ /home/iliya/deep_venv/bin/python -m pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless || true


wrote /home/iliya/deep_venv/lib/python3.12/site-packages/opencv_cuda.pth


## 5. Verify

Colab may print `ERROR:root:Error disabling cv2.imshow()` when `cv2` is imported. This is harmless: our build has no GUI module (`highgui`), so there is no `imshow` for Colab to patch. Show images with matplotlib instead.

In [6]:
import site
site.addsitedir(f"{PREFIX}/python")   # same effect as the .pth file, for this session
import cv2

print("cv2 file   :", cv2.__file__)
print("version    :", cv2.__version__)
print("CUDA devices:", cv2.cuda.getCudaEnabledDeviceCount())
cv2.cuda.printShortCudaDeviceInfo(cv2.cuda.getDevice())

info = cv2.getBuildInformation()
start = info.find("NVIDIA CUDA")
print(info[start:start + 400])
assert cv2.cuda.getCudaEnabledDeviceCount() > 0, "OpenCV was not built with CUDA or no GPU is visible" 

cv2 file   : /home/iliya/opencv-cuda/python/cv2/__init__.py
version    : 5.0.0
CUDA devices: 1
Device 0:  "NVIDIA GeForce RTX 5070 Ti"  16275Mb, sm_120, Driver/Runtime ver.13.40/13.0
NVIDIA CUDA:                   YES (ver 13.0, CUFFT CUBLAS FAST_MATH)
    NVIDIA GPU arch:             120
    NVIDIA PTX archs:

  OpenCL:                        YES (no extra features)
    Include path:                /home/iliya/opencv-cuda-src/opencv/3rdparty/include/opencl/1.2
    Link libraries:              Dynamic load

  ONNX Runtime:                  NO

  Python 3:
    Interpreter:     


### Quick smoke test: upload → filter → download

A GPU workflow always has the same shape: copy the image to GPU memory (`GpuMat.upload`), run filters there, copy the result back (`download`). The copies cost time, so the GPU pays off for large images or long chains of operations — not for a single small filter.

In [7]:
import numpy as np
img = (np.random.default_rng(0).random((1080, 1920)) * 255).astype(np.float32)
g = cv2.cuda.GpuMat(); g.upload(img)
blur = cv2.cuda.createGaussianFilter(cv2.CV_32FC1, cv2.CV_32FC1, (5, 5), 1.0)
out = blur.apply(g).download()
ref = cv2.GaussianBlur(img, (5, 5), 1.0)
print("max |GPU - CPU| =", float(np.abs(out - ref).max()))

max |GPU - CPU| = 4.57763671875e-05


## Running it locally (VS Code / Jupyter)

This notebook detects where it runs. On your own Linux or WSL2 machine with an NVIDIA driver and the CUDA Toolkit (`nvcc`), select a Python environment as the kernel and run all cells: sources go to `~/opencv-cuda-src`, the build to `~/opencv-cuda`, and no root access is needed. On native Windows, use the *x64 Native Tools* prompt with Visual Studio's generator instead of Ninja. Without an NVIDIA GPU there is nothing to accelerate — the Week 4 filter notebooks fall back to the CPU automatically.

Used later by **`week4/01_prewitt_x_gauss.ipynb`**, **`02_prewitt_y_gauss.ipynb`**, **`03_sobel_gauss.ipynb`** and **`05_laplacian_gauss.ipynb`**. They look for the build in `/opt/opencv-cuda` (Colab), in the Drive cache (Colab, `USE_DRIVE = True` here), in `~/opencv-cuda` (local), or in the folder named by the `OPENCV_CUDA_PREFIX` environment variable.